# Healthcare Operations Analytics — Python Analysis Notebook

## Purpose

This notebook performs Day 2 analysis for the Healthcare Operations Analytics mini project.

It uses the SQL-validated analytical mart exported from Phase 2:

`data/processed/healthcare_ed_throughput_mart.csv`

The analysis focuses on profiling ED throughput measures, benchmarking hospitals within ED-volume peer categories, and preparing outputs for visuals, findings, and portfolio packaging.

## Scope Guardrail

This notebook supports operational screening and benchmarking.

It does not perform causal modeling, hospital ranking, trauma-level adjustment, staffing analysis, or definitive quality diagnosis.

## 1. Load Phase 2 Analytical Mart

This section loads the SQL-validated hospital-level mart created during Phase 2.

Validation checks:

- row count
- unique facility count
- facility_id preservation
- core field availability

In [2]:
import pandas as pd
from pathlib import Path

# Notebook is inside /notebooks, so project root is one level up
project_root = Path.cwd().parent

processed_path = project_root / "data" / "processed"
mart_path = processed_path / "healthcare_ed_throughput_mart.csv"

df = pd.read_csv(
    mart_path,
    dtype={"facility_id": "string"}
)

print("Mart path:", mart_path)
print("Shape:", df.shape)
print("Unique facilities:", df["facility_id"].nunique())
print("First facility IDs:")
print(df["facility_id"].head())

display(df.head())

Mart path: C:\Projects\AnalyticsPortfolio\healthcare-operations-cms\data\processed\healthcare_ed_throughput_mart.csv
Shape: (4660, 22)
Unique facilities: 4660
First facility IDs:
0    010001
1    010005
2    010006
3    010007
4    010011
Name: facility_id, dtype: string


,facility_id,facility_name,city_town,state,hospital_type,hospital_ownership,emergency_services,ed_volume_category,op_18b_median_wait_min,op_22_lwbs_pct,...,op_22_availability_status,edv_footnote,op_18b_footnote,op_22_footnote,edv_start_date,edv_end_date,op_18b_start_date,op_18b_end_date,op_22_start_date,op_22_end_date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,Acute Care Hospitals,Government - Hospital District or Authority,Yes,very high,217.0,3.0,...,Available,NaN,NaN,NaN,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024
1,010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,Acute Care Hospitals,Government - Hospital District or Authority,Yes,very high,141.0,3.0,...,Available,NaN,NaN,NaN,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024
2,010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,Acute Care Hospitals,Proprietary,Yes,high,144.0,1.0,...,Available,NaN,NaN,NaN,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024
3,010007,MIZELL MEMORIAL HOSPITAL,OPP,AL,Acute Care Hospitals,Voluntary non-profit - Private,Yes,low,128.0,1.0,...,Available,NaN,NaN,NaN,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024
4,010011,ST. VINCENT'S EAST,BIRMINGHAM,AL,Acute Care Hospitals,Voluntary non-profit - Private,Yes,NaN,156.0,NaN,...,Not Available,5.0,"3, 29",5,01/01/2024,12/31/2024,07/01/2024,06/30/2025,01/01/2024,12/31/2024


In [3]:
core_fields = [
    "ed_volume_category",
    "op_18b_median_wait_min",
    "op_22_lwbs_pct"
]

missing_profile = pd.DataFrame({
    "field": core_fields,
    "missing_count": [df[field].isna().sum() for field in core_fields],
    "available_count": [df[field].notna().sum() for field in core_fields],
    "missing_pct": [df[field].isna().mean() * 100 for field in core_fields]
})

display(missing_profile)

,field,missing_count,available_count,missing_pct
0,ed_volume_category,823,3837,17.660944
1,op_18b_median_wait_min,583,4077,12.510730
2,op_22_lwbs_pct,828,3832,17.768240


In [4]:
complete_case_count = df[core_fields].dropna().shape[0]

print("Complete cases with EDV + OP_18b + OP_22:", complete_case_count)
print("Total hospitals:", len(df))
print("Complete-case percentage:", complete_case_count / len(df) * 100)

Complete cases with EDV + OP_18b + OP_22: 3777
Total hospitals: 4660
Complete-case percentage: 81.05150214592275


In [5]:
edv_distribution = (
    df["ed_volume_category"]
    .value_counts(dropna=False)
    .rename_axis("ed_volume_category")
    .reset_index(name="hospital_count")
)

edv_distribution["pct_of_total"] = (
    edv_distribution["hospital_count"] / len(df) * 100
)

display(edv_distribution)

,ed_volume_category,hospital_count,pct_of_total
0,low,1666,35.751073
1,medium,915,19.635193
2,NaN,823,17.660944
3,very high,704,15.107296
4,high,552,11.845494


In [6]:
edv_valid_distribution = (
    df.dropna(subset=["ed_volume_category"])
      ["ed_volume_category"]
      .value_counts()
      .rename_axis("ed_volume_category")
      .reset_index(name="hospital_count")
)

edv_valid_distribution["pct_of_valid_edv"] = (
    edv_valid_distribution["hospital_count"] 
    / edv_valid_distribution["hospital_count"].sum() 
    * 100
)

display(edv_valid_distribution)

,ed_volume_category,hospital_count,pct_of_valid_edv
0,low,1666,43.419338
1,medium,915,23.846755
2,very high,704,18.347667
3,high,552,14.386239


In [7]:
outcome_fields = [
    "op_18b_median_wait_min",
    "op_22_lwbs_pct"
]

outcome_summary = df[outcome_fields].describe().T

display(outcome_summary)

,count,mean,std,min,25%,50%,75%,max
op_18b_median_wait_min,4077.0,157.080942,51.264729,42.0,119.0,148.0,188.0,464.0
op_22_lwbs_pct,3832.0,1.684499,1.761531,0.0,1.0,1.0,2.0,23.0


In [8]:
percentile_summary = df[outcome_fields].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95]
).T

display(percentile_summary)

,0.25,0.50,0.75,0.90,0.95
op_18b_median_wait_min,119.0,148.0,188.0,225.0,250.0
op_22_lwbs_pct,1.0,1.0,2.0,4.0,5.0


In [9]:
edv_outcome_summary = (
    df
    .dropna(subset=["ed_volume_category"])
    .groupby("ed_volume_category")
    .agg(
        hospital_count=("facility_id", "count"),
        op_18b_available=("op_18b_median_wait_min", "count"),
        op_18b_median=("op_18b_median_wait_min", "median"),
        op_18b_p75=("op_18b_median_wait_min", lambda x: x.quantile(0.75)),
        op_18b_p90=("op_18b_median_wait_min", lambda x: x.quantile(0.90)),
        op_22_available=("op_22_lwbs_pct", "count"),
        op_22_median=("op_22_lwbs_pct", "median"),
        op_22_p75=("op_22_lwbs_pct", lambda x: x.quantile(0.75)),
        op_22_p90=("op_22_lwbs_pct", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

display(edv_outcome_summary)

,ed_volume_category,hospital_count,op_18b_available,op_18b_median,op_18b_p75,op_18b_p90,op_22_available,op_22_median,op_22_p75,op_22_p90
0,high,552,551,190.0,219.0,252.0,552,2.0,3.0,4.9
1,low,1666,1612,120.0,140.0,162.9,1661,1.0,1.0,3.0
2,medium,915,911,168.0,194.0,218.0,915,2.0,3.0,4.0
3,very high,704,703,194.0,231.0,271.6,704,2.0,3.0,5.0


## Block 6 Completion Checkpoint — Python Profiling

Block 6 is complete.

Completed work:

- Loaded the SQL-validated analytical mart from `data/processed/healthcare_ed_throughput_mart.csv`.
- Confirmed the mart contains 4,660 rows and 4,660 unique facilities.
- Confirmed `facility_id` is preserved as text with leading zeroes.
- Profiled missingness for the three core fields:
  - `ed_volume_category`
  - `op_18b_median_wait_min`
  - `op_22_lwbs_pct`
- Confirmed complete-case population for EDV + OP_18b + OP_22:
  - 3,777 hospitals
  - 81.1% of the analytical mart
- Profiled EDV category distribution.
- Profiled OP_18b and OP_22 distributions overall.
- Profiled OP_18b and OP_22 by EDV tier.

Key profiling results:

- EDV available: 3,837 hospitals
- OP_18b available: 4,077 hospitals
- OP_22 available: 3,832 hospitals
- Complete-case benchmarking population: 3,777 hospitals

EDV peer groups are large enough for descriptive benchmarking:

- low: 1,666 hospitals
- medium: 915 hospitals
- high: 552 hospitals
- very high: 704 hospitals

Outcome distribution summary:

- Overall median OP_18b: 148 minutes
- Overall 90th percentile OP_18b: 225 minutes
- Overall median OP_22: 1%
- Overall 90th percentile OP_22: 4%

EDV-tier profiling showed that OP_18b generally increases across ED volume categories, with higher-volume ED groups showing longer median wait times and higher upper-tail values.

Interpretation guardrail:

These results support descriptive profiling and peer benchmarking. They do not support causal claims, definitive hospital rankings, or statements that longer waits caused patients to leave before being seen.

Immediate next action:

Continue in `03_healthcare_peer_benchmarking.ipynb`.

The next block is Block 7 — EDV Peer Benchmarking.